# Explore videos with DBP API

Connect → inspect coverage → filter → optionally download assignments.
The website must expose `/api/v1`. No presentation framework is needed.
Run cells in order. Counts come from the server's loaded dataset, not this page of results.


In [ ]:
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display
from dbp_api import Client, Experiment, Assignments, Session, Trial, MetricFilter, Video
# from dbp_api import Image  # Future modality; not yet supported by the server.

website_url = input("Website URL [http://127.0.0.1:8773]: ").strip() or "http://127.0.0.1:8773"
client = Client(website_url, timeout=120)
client.login(input("Username: "), getpass("Password: "))
print("Connected")


## All metrics and coverage

`known` counts videos with a value, including zero or false. `unknown` means no
usable value. Each metric has its own coverage; adding these counts would double
count videos. The request returns aggregates for the full dataset, but only one
video row. No duration or content filter is applied here.


In [ ]:
inventory = client.metrics()
baseline = client.query_media(limit=1)
print(f"{baseline['matching']:,} of {baseline['total']:,} videos")

def metric_coverage(result):
    records = []
    for metric in inventory["metrics"]:
        summary = result["distributions"].get(metric["id"])
        known = summary["known"] if summary is not None else None
        records.append({
            "Metric": metric["label"],
            "ID": metric["id"],
            "Type": metric["kind"],
            "Unit": metric["unit"],
            "Operators": ", ".join(metric["operators"]),
            "Measured videos": known,
            "Unknown videos": summary["unknown"] if summary is not None else None,
            "Measured %": 100 * known / result["matching"] if known is not None and result["matching"] else None,
        })
    return pd.DataFrame(records)

with pd.option_context("display.max_rows", None):
    display(metric_coverage(baseline))


## Filter videos

Edit the content query and metric filters below. All constraints are combined.
Content search uses visual captions and spoken transcripts (`corpus="both"`);
choose `captions` or `transcripts` to restrict the source. An empty query matches
all content. Missing metric values do not satisfy numeric comparisons.


In [ ]:
content_query = ""
filters = [MetricFilter("duration_seconds", "gte", 10)]
query = dict(content_query=content_query, corpus="both", filters=filters, version=baseline["version"])
selected = client.query_media(**query, limit=20)
query["search_version"] = selected.get("search_version")
print(f"{selected['matching']:,} of {selected['total']:,} videos match; {len(selected['rows'])} shown")
display(pd.DataFrame(selected["rows"]))
with pd.option_context("display.max_rows", None):
    display(metric_coverage(selected))


## Inspect a distribution

These bins cover **all matching videos**, not just the displayed page. Baseline
and filtered counts use the same bin edges. Change the ID to any listed metric.


In [ ]:
metric_id = "duration_seconds"
distribution = selected["distributions"][metric_id]
display(pd.DataFrame(distribution["bins"]))
display(pd.DataFrame([distribution["baseline"], {
    key: distribution[key] for key in ("known", "unknown", "median", "minimum", "maximum")
}], index=["Full dataset", "Filtered"]))


## Create an experiment recipe

An `Experiment` pins the current dataset version, filters, and customizable string
seed. It is a local recipe until `assign()` saves assignments to the website.
`items_per_subject` counts original videos across all blocks; repeats and foils
are additional presentations. Images are shown below as a future extension, not
a working server feature.


In [ ]:
experiment: Experiment = client.create_experiment(
    name="API demo", seed="demo-1", filters=filters,
    content_query=content_query, corpus="both", media_type=Video,
)
print("Dataset version:", experiment.selection.version)

# Future image example, once the server supports image datasets:
# image_experiment = client.create_experiment(
#     name="Image study", seed="images-1", filters=[], media_type=Image,
# )
# image_assignments = image_experiment.assign(subjects=1, items_per_subject=2)


## Assign and download (optional)

Set `download_demo = True` to save and publish a two-video assignment. This writes
to the website and downloads files. Rerunning creates another assignment; use
`client.assignments(saved_id)` to reopen an existing one instead.
The download destination must be new. No media plays and no presentation is recorded.


In [ ]:
download_demo = False

if download_demo:
    assignments: Assignments = experiment.assign(
        subjects=1, items_per_subject=2, blocks=1, foils_per_block=0,
    )
    print("Save this experiment ID:", assignments.experiment_id)
    print("Subject IDs:", assignments.subject_ids)
    session: Session = assignments.subject(assignments.subject_ids[0], workspace=".dbp")
    trials: tuple[Trial, ...] = session.download(Path("dbp-demo-download"))
    display(pd.DataFrame([{
        "trial": trial.trial_id, "media": trial.media_id, "block": trial.block_index,
        "role": trial.role, "path": str(trial.local_path),
    } for trial in trials]))
else:
    print("Assignment and download skipped")

# Reopen saved assignments without resampling (also works on another device):
# assignments = client.assignments("PASTE_SAVED_EXPERIMENT_ID")
# session = assignments.subject("subject-001", workspace=".dbp")
# session.download("new-local-download-directory")


## Report presentation from your own task

`Trial` describes the media, block, role, optional foil `Segment`, and local path.
Call `started()` at presentation onset and `completed()` only after successful
playback. If `started()` fails, stop the task. For precise stimulus timing, perform
synchronization outside timing-critical display code; these calls involve network I/O.

Events are saved in a local SQLite journal before upload to the website. IDs make
upload retries safe. `pending_trials` excludes both started and completed trials;
`incomplete_trials` exposes started-but-unfinished trials for review, not automatic
replay. Progress is per trial, so intentional repeats remain separate.
Another device sees only progress that reached the server. Keep the journal after
an interruption and call `sync()` to retry uploads before switching devices.

The loop is commented out so running this notebook cannot falsely record playback.


In [ ]:
# Your presentation framework supplies present_video(), which returns only
# after successful playback and raises if presentation is interrupted.
#
# for trial in session.pending_trials:
#     session.started(trial)
#     present_video(trial.local_path)
#     session.completed(trial)
# session.sync()
#
# Review interrupted trials before continuing an experiment:
# display(session.incomplete_trials)
# print(session.journal_path)
#
# Future image task: use present_image(trial.local_path) instead of present_video.
# Presentation code stays outside dbp_api; progress calls stay the same.


## Disconnect
Run this when finished. Clear outputs before sharing.


In [ ]:
client.logout()
print("Disconnected")
